### RAG Pipeline - Data Ingestion to Vector DB

In [1]:
import os
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

/Users/aashitpaliwal/Desktop/RAGTEST/ragvenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
## Read all pdf's inside directory
def processpdfs(pdf_directory):
  all_documents = []
  pdf_dir = Path(pdf_directory)

  pdf_files = list(pdf_dir.glob("**/*.pdf"))

  print(f"found {len(pdf_files)} PDF files to process")

  for pdf_file in pdf_files:
    print(f"\nProcessing: {pdf_file.name}")
    try:
      loader = PyMuPDFLoader(str(pdf_file))
      documents = loader.load()

      for doc in documents:
        doc.metadata['source_file'] = pdf_file.name
        doc.metadata['file_type'] = 'pdf'
      
      all_documents.extend(documents)
      print(f"Loaded {len(documents)} pages")
    
    except Exception as e:
      print(f"Error: {e}")
  
  print(f"\nTotal documents loaded: {len(all_documents)}")
  return all_documents

all_pdf_documents = processpdfs("../data")

found 4 PDF files to process

Processing: Frank Guridy.pdf
Loaded 45 pages

Processing: Gail Saunders.pdf
Loaded 23 pages

Processing: Florence Babb.pdf
Loaded 30 pages

Processing: Henrice Altink.pdf
Loaded 35 pages

Total documents loaded: 133


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'IthakaSplitter; modified using iText® 7.1.3 ©2000-2018 iText Group NV (JSTOR Michigan; licensed version)', 'creator': 'IthakaSplitter', 'creationdate': '2012-09-17T05:36:55-04:00', 'source': '../data/pdf/Frank Guridy.pdf', 'file_path': '../data/pdf/Frank Guridy.pdf', 'total_pages': 45, 'format': 'PDF 1.6', 'title': 'Destination without Humiliation:', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2020-09-02T19:04:57+00:00', 'trapped': '', 'modDate': 'D:20200902190457Z', 'creationDate': "D:20120917053655-04'00'", 'page': 0, 'source_file': 'Frank Guridy.pdf', 'file_type': 'pdf'}, page_content='Chapter Title: Destination without Humiliation: Black Travel within the Routes of \nDiscrimination\n \n \nBook Title: Forging Diaspora \nBook Subtitle: Afro-Cubans and African Americans in a World of Empire and Jim Crow \nBook Author(s): Frank Andre Guridy \nPublished by: University of North Carolina Press \nStable URL: https://www.jstor.org/stable/10.5149

In [4]:
## Text splitting into chunks

from sqlalchemy import text


def split_documents(documents, chunk_size=1000, chunk_overlap=200):
  """Split documents into smaller chunks for better rag performance"""
  text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
  )

  split_docs = text_splitter.split_documents(documents)
  print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

  # Show example of a chunk
  if split_docs:
    print(f"\nExample chunk:")
    print(f"Content: {split_docs[0].page_content[:200]}...")
    print(f"Metadata: {split_docs[0].metadata}")
  
  return split_docs


In [5]:
chunks=split_documents(all_pdf_documents)

Split 133 documents into 490 chunks

Example chunk:
Content: Chapter Title: Destination without Humiliation: Black Travel within the Routes of 
Discrimination
 
 
Book Title: Forging Diaspora 
Book Subtitle: Afro-Cubans and African Americans in a World of Empir...
Metadata: {'producer': 'IthakaSplitter; modified using iText® 7.1.3 ©2000-2018 iText Group NV (JSTOR Michigan; licensed version)', 'creator': 'IthakaSplitter', 'creationdate': '2012-09-17T05:36:55-04:00', 'source': '../data/pdf/Frank Guridy.pdf', 'file_path': '../data/pdf/Frank Guridy.pdf', 'total_pages': 45, 'format': 'PDF 1.6', 'title': 'Destination without Humiliation:', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2020-09-02T19:04:57+00:00', 'trapped': '', 'modDate': 'D:20200902190457Z', 'creationDate': "D:20120917053655-04'00'", 'page': 0, 'source_file': 'Frank Guridy.pdf', 'file_type': 'pdf'}


## Embeddings and Vector DB

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
class EmbeddingManager:
  def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
    """
    Initialize embedding manager

    Args:
      model_name: HuggingFace model name for sentence embeddings
    """
    self.model_name = model_name
    self.model = None
    self._load_model()
  
  def _load_model(self):
    """Load sentence transformer model"""
    try:
      print(f"Loading embedding model: {self.model_name}")
      self.model = SentenceTransformer(self.model_name)
      print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
    except Exception as e:
      print(f"Error in loading model {self.model_name}: {e}")
      raise
  
  def generate_embeddings(self, texts: List[str]) -> np.ndarray:
    """
    Generate embeddings for a list fo texts

    Args:
      texts: List of text strings to embed
    
    Returns:
      numpy array of embeddings with shape (len(texts), embedding_dim)
    """

    if not self.model:
      raise ValueError("Model not loaded")
    
    print(f"Generating embeddings for {len(texts)} texts ...")
    embeddings = self.model.encode(texts, show_progress_bar=True)
    print(f"Generated embeddings with shape {embeddings.shape}")
    return embeddings

## Initialize embedding manager

embedding_manager = EmbeddingManager()
embedding_manager


Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13112.37it/s]


Model loaded successfully. Embedding dimension: 384


## Vector Store

In [8]:
from curses import meta


class VectorStore:

  def __init__ (self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
    """
    Initialize vector store

    Args:
      collection_name: Name of ChromaDB collection
      persist_directory: Directory to persist the Vector store
    """

    self.collection_name = collection_name
    self.persist_directory = persist_directory
    self.client = None
    self.collection = None
    self._initialize_store()
  
  def _initialize_store(self):
    """Initialize ChromaDb client and collection"""
    try:
      os.makedirs(self.persist_directory, exist_ok=True)
      self.client = chromadb.PersistentClient(path=self.persist_directory)

      self.collection = self.client.get_or_create_collection(
        name=self.collection_name,
        metadata={"description": "PDF document embeddinsg for RAG"}
      )

      print(f"Vector store initialized. Collection: {self.collection_name}")
      print(f"Existing documents in collection: {self.collection.count()}")

    except Exception as e:
      print(f"Error initializing vector store: {e}")
      raise
  
  def add_documents(self, documents: List[Any], embeddings: np.ndarray):
    """
    Add documents and their embeddings to the vector store
    
    Args:
      documents: List of langChain documents
      embeddings: Corresponding embeddings for the documents
    """

    if len(documents) != len(embeddings):
      raise ValueError("Num docs don't match num embeddings")
    
    print(f"Adding {len(documents)} documents to the vector store...")

    ids= []
    metadatas = []
    documents_text = []
    embeddings_list = []

    for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
      # Generate unique id
      doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
      ids.append(doc_id)

      # Prepare metadata
      metadata = dict(doc.metadata)
      metadata['doc_index'] = i
      metadata['content_length'] = len(doc.page_content)
      metadatas.append(metadata)

      # Document content
      documents_text.append(doc.page_content)

      # Embedding
      embeddings_list.append(embedding.tolist())
    
    # Add to collection
    try:
      self.collection.add(
        ids=ids,
        embeddings=embeddings_list,
        metadatas=metadatas,
        documents=documents_text
      )
      print(f"Successfully added {len(documents)} documents to vector store")
      print(f"Total documents in collection {self.collection.count()}")
    
    except Exception as e:
      print(f"Error addings documents to vector store: {e}")
      raise

vectorstore=VectorStore()
vectorstore


Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [9]:
chunks

[Document(metadata={'producer': 'IthakaSplitter; modified using iText® 7.1.3 ©2000-2018 iText Group NV (JSTOR Michigan; licensed version)', 'creator': 'IthakaSplitter', 'creationdate': '2012-09-17T05:36:55-04:00', 'source': '../data/pdf/Frank Guridy.pdf', 'file_path': '../data/pdf/Frank Guridy.pdf', 'total_pages': 45, 'format': 'PDF 1.6', 'title': 'Destination without Humiliation:', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2020-09-02T19:04:57+00:00', 'trapped': '', 'modDate': 'D:20200902190457Z', 'creationDate': "D:20120917053655-04'00'", 'page': 0, 'source_file': 'Frank Guridy.pdf', 'file_type': 'pdf'}, page_content='Chapter Title: Destination without Humiliation: Black Travel within the Routes of \nDiscrimination\n \n \nBook Title: Forging Diaspora \nBook Subtitle: Afro-Cubans and African Americans in a World of Empire and Jim Crow \nBook Author(s): Frank Andre Guridy \nPublished by: University of North Carolina Press \nStable URL: https://www.jstor.org/stable/10.5149

In [10]:
# Convert text to embeddings 
texts = [doc.page_content for doc in chunks]

# Generate the embeddings
embeddings = embedding_manager.generate_embeddings(texts)

# Store in vector db
vectorstore.add_documents(chunks, embeddings)


Generating embeddings for 490 texts ...


Batches: 100%|██████████| 16/16 [00:02<00:00,  7.47it/s]


Generated embeddings with shape (490, 384)
Adding 490 documents to the vector store...
Successfully added 490 documents to vector store
Total documents in collection 490


## Retriever pipeline from VectorStore

In [11]:
from importlib import metadata
from turtle import distance


class RAGRetreiver:
  """Handles query-based retreival from the vector store"""

  def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
    """
    Initialize the retreiver

    Args:
      vector_store: Vector store with document embeddings
      embedding_manager: Manager for generating query embeddings
    """
    self.vector_store = vector_store
    self.embedding_manager = embedding_manager
  
  def retreive(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
    """
    Retreive relevant documents for a query

    Args:
      query: Search query
      top_k: Number of top results to return
      score_threshold: Minimum similarity score threshold
    
    Returns:
      List of dictionaries containing retreived documents and metadata
    """
    print(f"Retreiving documents for query: {query}")
    print(f"Top K: {top_k}, Score Threshold: {score_threshold}")

    # Generate query embedding 
    query_embedding = self.embedding_manager.generate_embeddings([query])[0]

    # Search vector store
    try:
      results = self.vector_store.collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=top_k
      )

      retreived_docs = []

      if results['documents'] and results['documents'][0]:
        documents = results['documents'][0]
        metadatas = results['metadatas'][0]
        distances = results['distances'][0]
        ids = results['ids'][0]

        for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
          similarity_score = 1 - distance

          if similarity_score >= score_threshold:
            retreived_docs.append({
              'id': doc_id,
              'content': document,
              'metadata': metadata,
              'similarity_score': similarity_score,
              'distance': distance,
              'rank': i + 1
            })
        
        print(f"Retreived {len(retreived_docs)} documents (after filtering)")
      else:
        print(f"No documents found")
      
      return retreived_docs
    
    except Exception as e:
      print(f"Error during retreival: {e}")
      return []

rag_retreiver = RAGRetreiver(vectorstore, embedding_manager)

In [12]:
rag_retreiver

In [13]:
rag_retreiver.retreive("Explain Cuban Revolution", )

Retreiving documents for query: Explain Cuban Revolution
Top K: 5, Score Threshold: 0.0
Generating embeddings for 1 texts ...


Batches: 100%|██████████| 1/1 [00:00<00:00, 17.57it/s]

Generated embeddings with shape (1, 384)
Retreived 5 documents (after filtering)


[{'id': 'doc_0bd4355d_69',
  'content': 'this time in Cuba.\n \nTh e Cuba that Mitchell encountered in 1937 was in the midst of a political \ntransition from authoritarian rule to a new political system under a new consti-\ntution that would be implemented a few years later. Th e political transforma-\ntions of the 1930s had an enormous impact on antidiscrimination activists. Th e \nfall of the Machado regime in 1933 helped stimulate a renewed movement for \nracial equality on the island. During the 1930s, a younger generation of male \nand female activists with roots in the Afro- Cuban societies and the Cuban \nCommunist Party waged a battle for the citizenship rights belonging to Afro-\n Cubans. Younger activists such as Salvador García Agüero and Nicolás Guillén, \namong others, challenged the Afro- Cuban leadership class that was associated \nwith the hated Machado, including some of the more prominent leaders of the \nClub Atenas. Th ey insisted that more active measures be taken 

In [14]:
## Simple RAG pipeline with groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")
if not groq_api_key:
  raise ValueError("GROQ_API_KEY not found in environment variables")

llm = ChatGroq(api_key=groq_api_key, model="llama-3.1-8b-instant", temperature=0.1, max_tokens=1024)

## RAG Function
def rag_simple(query, retreiver, llm, top_k=3):
  results = retreiver.retreive(query, top_k=top_k)
  context = "\n\n".join([doc['content'] for doc in results]) if results else ""
  if not context:
    return "No relevant context found"

  ## generate answer
  prompt = f"Use the following context to answer the question concisely:\n\nContext:\n{context}\n\nQuestion: {query}\nAnswer:"

  response = llm.invoke([prompt])
  return response.content

In [15]:
answer = rag_simple("Explain African activism for travel", rag_retreiver, llm)
print(answer)

Retreiving documents for query: Explain African activism for travel
Top K: 3, Score Threshold: 0.0
Generating embeddings for 1 texts ...


Batches: 100%|██████████| 1/1 [00:00<00:00, 33.51it/s]

Generated embeddings with shape (1, 384)
Retreived 3 documents (after filtering)


African activism for travel in the context of the Afro-diasporic experience involved challenging racial exclusion and humiliation faced by African descendants in the United States and Cuba. This activism took shape in the 1930s, with the emergence of organizations and movements that advocated for the right to travel and leisure without discrimination. African American entrepreneurs also developed their own tourist networks, relying on segregated institutions and businesses that catered to African American travelers, as a form of adaptation and counter-strategy to imperial structures governing travel and leisure.


## Enhanced RAG Pipeline

In [16]:
def rag_advanced(query, retreiver, llm, top_k = 5, min_score = 0.2, return_context = False):
  """
  Advanced RAG Pipeline
   - Returns answer, sources, confidence score and optionally full context
  """
  results = retreiver.retreive(query, top_k=top_k, score_threshold=min_score)
  if not results:
    return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
  
  context = "\n\n".join([doc['content'] for doc in results])
  sources = [{
    'sources': doc['metadata'].get('source_file', 'unknown'),
    'page': doc['metadata'].get('page', 'unknown'),
    'similarity_score': doc['similarity_score'],
    'preview': doc['content'][:120] + "..." if len(doc['content']) > 120 else doc['content']
  } for doc in results]
  confidence = max(doc['similarity_score'] for doc in results)

  #Generate answer
  prompt = f"Use the following context to answer the question concisely:\n\nContext:\n{context}\n\nQuestion: {query}\nAnswer:"
  response  = llm.invoke([prompt.format(context=context, query=query)])

  output = {
    'answer': response.content,
    'sources': sources,
    'confidence': confidence
  }
  if return_context:
    output['context'] = context
  return output

In [17]:
result = rag_advanced("Explain african activism for travel", rag_retreiver, llm, top_k=3, min_score=0.1, return_context=True)
print(f"Answer: {result['answer']}")
print(f"Sources: {result['sources']}")
print(f"Confidence Score: {result['confidence']}")
print(f"Context: {result['context'][:300]}...")

Retreiving documents for query: Explain african activism for travel
Top K: 3, Score Threshold: 0.1
Generating embeddings for 1 texts ...


Batches: 100%|██████████| 1/1 [00:00<00:00, 19.54it/s]

Generated embeddings with shape (1, 384)
Retreived 3 documents (after filtering)


Answer: African activism for travel in the context of the Afro-diasporic experience involved challenging racial exclusion and humiliation faced by African Americans while traveling. This activism took shape in the 1930s, with the emergence of black radical organizations and the international Scottsboro campaign. African American entrepreneurs also developed their own transnational strategies to counter racial exclusion by creating tourist networks that catered to African American travelers, often relying on segregated institutions and businesses.
Sources: [{'sources': 'Frank Guridy.pdf', 'page': 6, 'similarity_score': 0.22730612754821777, 'preview': '156 \nDestination without Humiliation\nrooms, and garage accommodations” in over three hundred cities in the United \nState...'}, {'sources': 'Frank Guridy.pdf', 'page': 2, 'similarity_score': 0.227073073387146, 'preview': 'created opportunities for black entrepreneurs to assemble a tourist network \nthat relied upon segregated institution

## Advanced RAG Pipeline

In [18]:
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
  def __init__(self, retreiver, llm):
    self.retreiver = retreiver
    self.llm = llm
    self.history = [] # Store query History

  def query(self, question: str, top_k: int=5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:

    results = self.retreiver.retreive(question, top_k=top_k, score_threshold=min_score)
    if not results:
      answer = "No relevant context found."
      sources = []
      context = ""
    else:
      context = "\n\n".join([doc['content'] for doc in results])
      sources = [{
        'sources': doc['metadata'].get('source_file', 'unknown'),
        'page': doc['metadata'].get('page', 'unknown'),
        'similarity_score': doc['similarity_score'],
        'preview': doc['content'][:120] + "..." if len(doc['content']) > 120 else doc['content']
      } for doc in results]

      prompt = f"Use the following context to answer the question concisely:\n\nContext:\n{context}\n\nQuestion: {question}\nAnswer:"
      
      if stream:
        print("Streaming response:")
        for i in range(0, len(prompt), 80):
          print(prompt[i:i+80], end="", flush=True)
          time.sleep(0.05) # Simulate streaming delay
          print()
      response = self.llm.invoke([prompt.format(context=context, question=question)])
      answer = response.content

    # Add citations to answer
    citations = [f"[{i+1}] {src['sources']} (Page: {src['page']})" for i, src in enumerate(sources)]
    answer_with_citations = answer + "\n\nSources:\n" + "\n".join(citations) if citations else answer

    # Summarize answer
    summary = None
    if summarize and answer:
      summary_prompt = f"Summarize the following answer in 2-3 sentences:\n\n{answer}"
      summary_response = self.llm.invoke([summary_prompt])
      summary = summary_response.content
    
    # Store in history
    self.history.append({
      'question': question,
      'answer': answer,
      'sources': sources,
      'summary': summary
    })

    return {
      'question': question,
      'answer': answer_with_citations,
      'sources': sources,
      'summary': summary,
      'history': self.history
    }

adv_rag = AdvancedRAGPipeline(rag_retreiver, llm)
result = adv_rag.query("Explain african activism for travel", top_k=3, min_score=0.1, stream=True, summarize=True)
print(f"\nFinal Answer: {result['answer']}")
print(f"Summary: {result['summary']}")
print(f"History: {result['history'][-1]}")

Retreiving documents for query: Explain african activism for travel
Top K: 3, Score Threshold: 0.1
Generating embeddings for 1 texts ...


Batches: 100%|██████████| 1/1 [00:00<00:00, 65.81it/s]

Generated embeddings with shape (1, 384)
Retreived 3 documents (after filtering)
Streaming response:
Use the following context to answer the question concisely:

Context:
156 
Desti
nation without Humiliation
rooms, and garage accommodations” in over three hundr
ed cities in the United 
States and Canada. Th at same year, another tourist com
pany advertised the 
benefi ts of automobile travel to “Negro travelers,” enabli


ng them to evade the 
humiliating experiences of Jim Crow railway cars.7
 
Th e 
emerging activism over the right to travel by African descendants in 
Cuba and t
he United States took shape in an age of heightened mobilization 
throughout the
 diaspora. Th e international Scottsboro campaign, the emerg-
ing anticolonial m
ovement in Africa and Europe that coalesced around the 
Italian invasion of Ethi
opia, and the explosion of militant labor uprisings in the 
Caribbean helped set
 in motion more aggressive challenges to ongoing prac-
tices of racial exclusion
.8 While black radical organizations were undeniably 
central to the forging of 
Afro- diasporic relationships in this period, the sup-

created opportunities fo
r black entrepreneurs to assemble a tourist network 
that relied upon segregated
 institutions and businesses that catered to Afri-
can American travelers. Th us
, what follows is not simply a tale of black bod-
ies experiencing discriminatio
n but a continuation of the